# Registering Custom RLSSM Models in HSSM

**Recommended:** define learning in `ssms.rl` and bridge with `from_ssms_model` (see [Advanced](rlssm_advanced.ipynb)).

When you want a **named, reusable template** entirely on the HSSM side, use `register_rlssm_model()` and `get_rlssm_model_config()`.

This notebook shows the registry path with the built-in `2AB_RescorlaWagner_Angle` template, then peeks at the manual `RLSSMConfig` fields.

## Setup

In [1]:

import logging
import os
import warnings

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import hssm
import ssms.rl as rl

warnings.filterwarnings("ignore")
logging.getLogger("jax._src.xla_bridge").setLevel(logging.ERROR)

hssm.set_floatX("float32", update_jax=True)


def make_participant_theta(group_theta, sds, bounds, n, rng):
    """Sample participant-level parameters around group means."""
    theta_arrays = {}
    for name, mean in group_theta.items():
        lo, hi = bounds[name]
        theta_arrays[name] = np.clip(rng.normal(mean, sds[name], size=n), lo, hi)
    true_df = pd.DataFrame(theta_arrays)
    true_df.index.name = "participant_id"
    return theta_arrays, true_df


PARTICIPANT_EFFECT_PRIOR = {
    "name": "Normal",
    "mu": {"name": "Normal", "mu": 0, "sigma": 0.1},
    "sigma": {"name": "HalfNormal", "sigma": 0.5},
}


def hierarchical_param(name, lo, hi, mu, sigma):
    """Build a truncated-normal group mean with participant random effects."""
    return hssm.Param(
        name,
        formula=f"{name} ~ 1 + (1|participant_id)",
        prior={
            "Intercept": hssm.Prior("TruncatedNormal", lower=lo, upper=hi, mu=mu, sigma=sigma),
            "1|participant_id": PARTICIPANT_EFFECT_PRIOR,
        },
    )


def per_participant_posterior_means(idata, list_params):
  post = idata.posterior
  out = {}
  for name in list_params:
      inter = post[f"{name}_Intercept"]
      re = post[f"{name}_1|participant_id"]
      re_dim = [d for d in re.dims if d not in ("chain", "draw")][0]
      per_p = (inter + re).mean(("chain", "draw"))
      ids = [int(v) for v in re[re_dim].values]
      out[name] = pd.Series(np.asarray(per_p.values), index=ids).sort_index()
  return pd.DataFrame(out)


def plot_group_recovery(idata, list_params, group_theta, title):
    names = [f"{n}_Intercept" for n in list_params]
    summ = az.summary(idata, var_names=names, kind="stats")
    summ.index = list_params
    summ["true"] = [group_theta[n] for n in list_params]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    y = np.arange(len(list_params))
    ax.errorbar(
        summ["mean"],
        y,
        xerr=[summ["mean"] - summ["hdi_3%"], summ["hdi_97%"] - summ["mean"]],
        fmt="o",
        capsize=4,
        label="posterior (intercept)",
    )
    ax.scatter(summ["true"], y, color="crimson", marker="D", zorder=5, label="true group mean")
    ax.set_yticks(y)
    ax.set_yticklabels(list_params)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    plt.show()
    return summ


def run_ppc(ssms_config, observed_data, ppc_theta, random_state=0):
    """Posterior predictive simulation conditioned on observed learning history."""
    return rl.Simulator(ssms_config).simulate(
        theta=ppc_theta,
        mode="ppc",
        observed_data=observed_data,
        random_state=random_state,
    )


def plot_rt_choice_ppc(observed, ppc, title="Posterior predictive check"):
    obs_rt = observed["rt"].to_numpy()
    ppc_rt = ppc["rt"].to_numpy()
    obs_rt = obs_rt[np.isfinite(obs_rt) & (obs_rt > 0)]
    ppc_rt = ppc_rt[np.isfinite(ppc_rt) & (ppc_rt > 0)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    if len(obs_rt) and len(ppc_rt):
        axes[0].hist(obs_rt, bins=25, density=True, alpha=0.5, label="observed")
        axes[0].hist(ppc_rt, bins=25, density=True, alpha=0.5, label="ppc")
    axes[0].set_title("RT distribution")
    axes[0].legend()
    obs_resp = observed["response"]
    ppc_resp = ppc["response"]
    valid = obs_resp > -900
    obs_choice = obs_resp[valid].value_counts(normalize=True).sort_index()
    valid_ppc = ppc_resp > -900
    ppc_choice = ppc_resp[valid_ppc].value_counts(normalize=True).sort_index()
    labels = sorted(set(obs_choice.index) | set(ppc_choice.index))
    x = np.arange(len(labels))
    axes[1].bar(x - 0.2, [obs_choice.get(c, 0) for c in labels], width=0.4, label="observed")
    axes[1].bar(x + 0.2, [ppc_choice.get(c, 0) for c in labels], width=0.4, label="ppc")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels)
    axes[1].set_title("Response proportions")
    axes[1].legend()
    fig.suptitle(title)
    plt.show()

from hssm.rl import get_rlssm_model_config, list_models, register_rlssm_model
from hssm.rl.registry import _compute_v_annotated


Setting PyTensor floatX type to float32.


Setting "jax_enable_x64" to False. If this is not intended, please set `jax` to False.


In [2]:
FULL_RUN = os.environ.get("FULL_RUN", "0") == "1"
N_PARTICIPANTS = 8 if FULL_RUN else 4
N_TRIALS = 120 if FULL_RUN else 60
RANDOM_SEED = 20260707


## Built-in registry models

In [3]:
print("Registered RLSSM models:", list_models())
config_builtin = get_rlssm_model_config("2AB_RescorlaWagner_Angle")
print("decision_process:", config_builtin.decision_process)
print("computed:", set(config_builtin.ssm_logp_func.computed))

Registered RLSSM models: {'2AB_RescorlaWagner_DDM': 'RLSSM model with Rescorla-Wagner Q-learning and the standard DDM as decision process.', '2AB_RescorlaWagner_Angle': 'RLSSM model with Rescorla-Wagner Q-learning and a collapsing-bound DDM (angle model) as decision process.', '2AB_RescorlaWagner_Weibull': 'RLSSM model with Rescorla-Wagner Q-learning and a Weibull-bound DDM as decision process.'}
decision_process: angle
computed: {'v'}


## Register a custom named model (recommended HSSM-side path)

Register once; reuse via `get_rlssm_model_config("my_angle_bandit")`. Here we register a copy of the standard angle RLSSM under a tutorial-specific name.

In [4]:
TUTORIAL_MODEL = "tutorial_2AB_RW_Angle"
if TUTORIAL_MODEL not in list_models():
    register_rlssm_model(
        name=TUTORIAL_MODEL,
        decision_process="angle",
        learning_process={"v": _compute_v_annotated},
        learning_process_params=["rl_alpha", "scaler"],
        learning_process_bounds={"rl_alpha": (0.0, 1.0), "scaler": (0.0, 10.0)},
        learning_process_params_default=[0.2, 2.0],
        extra_fields=["feedback"],
        choices=[-1, 1],
        description="Tutorial registration of standard RW + angle RLSSM",
    )
config_reg = get_rlssm_model_config(TUTORIAL_MODEL)
config_reg

RLSSMConfig(model_name='tutorial_2AB_RW_Angle', description='Tutorial registration of standard RW + angle RLSSM', response=['rt', 'response'], choices=(-1, 1), list_params=['rl_alpha', 'scaler', 'a', 'z', 't', 'theta'], bounds={'rl_alpha': (0.0, 1.0), 'scaler': (0.0, 10.0), 'a': (0.3, 3.0), 'z': (0.1, 0.9), 't': (0.001, 2.0), 'theta': (-0.1, 1.3)}, loglik=None, loglik_kind='approx_differentiable', backend=None, extra_fields=['feedback'], rv=None, decision_process_loglik_kind='approx_differentiable', learning_process_kind='blackbox', params_default=[0.2, 2.0, 1.65, 0.5, 1.0005, 0.6], decision_process='angle', learning_process={'v': <function compute_v_subject_wise at 0x125e16160>}, ssm_logp_func=<function make_jax_matrix_logp_funcs_from_onnx.<locals>.logp at 0x108bdb420>)

## Simulate data and fit via registry config

In [5]:
ssms_config = rl.preset.get("2AB_RW_Angle")
theta = {'rl_alpha': 0.25, 'scaler': 2.0, 'a': 1.2, 'z': 0.5, 't': 0.25, 'theta': 0.3}
data = rl.Simulator(ssms_config).simulate(
    theta=theta, n_trials=N_TRIALS, n_participants=N_PARTICIPANTS, random_state=RANDOM_SEED,
)
model = hssm.RLSSM(
    data=data,
    model_config=config_reg,
    p_outlier=0,
    lapse=None,
    include=[
        hierarchical_param("rl_alpha", 0.01, 1.0, 0.25, 0.15),
        hierarchical_param("scaler", 0.1, 5.0, 2.0, 0.8),
        hierarchical_param("a", 0.3, 2.5, 1.1, 0.3),
        hierarchical_param("z", 0.1, 0.9, 0.5, 0.15),
        hierarchical_param("t", 0.05, 1.0, 0.25, 0.1),
        hierarchical_param("theta", 0.0, 1.2, 0.30, 0.15),
    ],
)
idata = model.sample(draws=40, tune=40, chains=2, cores=2, sampler='numpyro', random_seed=RANDOM_SEED)
az.summary(idata, var_names=['rl_alpha_Intercept', 'scaler_Intercept'])

You supplied a model 'tutorial_2AB_RW_Angle', which is currently not supported in the ssm_simulators package. An error will be thrown when sampling from the random variable or when using any posterior or prior predictive sampling methods.


User prior for '1|participant_id' on parameter 'rl_alpha' supplies a hyperprior on `mu`, but the effective parameterization is non-centered. bambi will reparameterize this term as `offset * sigma` and drop the `mu` hyperprior, leaving it as a disconnected node in the PyMC graph. Either pass `noncentered=False` to `HSSM(...)` so that `mu` is used in the centered Normal, or move the location prior to the common `Intercept` (e.g. use a formula like 'rl_alpha ~ 1 + (1|participant_id)' and attach the `mu` prior to 'Intercept'). To silence this warning without changing the model, set the `mu` argument to a scalar (e.g. `mu=0`).


User prior for '1|participant_id' on parameter 'scaler' supplies a hyperprior on `mu`, but the effective parameterization is non-centered. bambi will reparameterize this term as `offset * sigma` and drop the `mu` hyperprior, leaving it as a disconnected node in the PyMC graph. Either pass `noncentered=False` to `HSSM(...)` so that `mu` is used in the centered Normal, or move the location prior to the common `Intercept` (e.g. use a formula like 'scaler ~ 1 + (1|participant_id)' and attach the `mu` prior to 'Intercept'). To silence this warning without changing the model, set the `mu` argument to a scalar (e.g. `mu=0`).


User prior for '1|participant_id' on parameter 'a' supplies a hyperprior on `mu`, but the effective parameterization is non-centered. bambi will reparameterize this term as `offset * sigma` and drop the `mu` hyperprior, leaving it as a disconnected node in the PyMC graph. Either pass `noncentered=False` to `HSSM(...)` so that `mu` is used in the centered Normal, or move the location prior to the common `Intercept` (e.g. use a formula like 'a ~ 1 + (1|participant_id)' and attach the `mu` prior to 'Intercept'). To silence this warning without changing the model, set the `mu` argument to a scalar (e.g. `mu=0`).


User prior for '1|participant_id' on parameter 'z' supplies a hyperprior on `mu`, but the effective parameterization is non-centered. bambi will reparameterize this term as `offset * sigma` and drop the `mu` hyperprior, leaving it as a disconnected node in the PyMC graph. Either pass `noncentered=False` to `HSSM(...)` so that `mu` is used in the centered Normal, or move the location prior to the common `Intercept` (e.g. use a formula like 'z ~ 1 + (1|participant_id)' and attach the `mu` prior to 'Intercept'). To silence this warning without changing the model, set the `mu` argument to a scalar (e.g. `mu=0`).


...

The number of samples is too small to check convergence reliably.


  0%|          | 0/80 [00:00<?, ?it/s]

 25%|██▌       | 20/80 [00:00<00:00, 199.67it/s]

100%|██████████| 80/80 [00:00<00:00, 646.19it/s]

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
rl_alpha_Intercept,0.505,0.0,0.505,0.505,0.0,0.0,80.0,80.0,NaN
scaler_Intercept,2.550,0.0,2.550,2.550,0.0,0.0,80.0,80.0,NaN


## Under the hood: manual `RLSSMConfig` fields

A bridged config from `from_ssms_model` populates the same slots: `ssm_logp_func` (decision likelihood + `.computed`), `learning_process`, `extra_fields`, and `list_params`. You rarely need to assemble these manually if you start from `ssms.rl` or the registry.

In [6]:
bridged = hssm.rl.RLSSMConfig.from_ssms_model(ssms_config)
print("Registry vs bridge list_params match:",
      config_reg.list_params == bridged.list_params)
print("Registry extra_fields:", config_reg.extra_fields)
print("Bridge extra_fields  :", bridged.extra_fields)

Registry vs bridge list_params match: True
Registry extra_fields: ['feedback']
Bridge extra_fields  : ['feedback']


HSSM also supports choice-only RL models (without RT). Those are documented separately once validated for the current release.